# EECE 5610, Chapter 12: root locus and frequency response of sampled-data loops.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

The running example of the chapter: a zero-order hold in front of Gp(s) = 1/(s(s+1)),
sampled with T = 1 s, a gain K and unity feedback.

This script reproduces the main numbers and figures of the chapter:
  1. G(z) by zero-order-hold discretization,
  2. the root locus in the z-plane: breakaway points, the circle, the critical gain,
  3. the gain that gives zeta = 0.5,
  4. the critical gain for T = 0.1, 0.5, 1 s,
  5. the w-plane transfer function and its Bode plot, with gain and phase margins,
  6. the Nyquist plot and the encirclement count Z = N + P,
  7. the root locus with one sample of computation delay, D(z) = K/z.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
import os

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.optimize import brentq

OUT = os.path.join(os.getcwd(), "ch12_figures")
os.makedirs(OUT, exist_ok=True)

In [ ]:
# -----------------------------------------------------------------------------------------
# 1. The pulse transfer function G(z) = (1 - z^-1) Z[Gp(s)/s]
# -----------------------------------------------------------------------------------------
def zoh_model(T):
    """Return (num, den) of G(z) for a ZOH in front of 1/(s(s+1)), coefficients in z."""
    num, den, _ = signal.cont2discrete(([1.0], [1.0, 1.0, 0.0]), T, method="zoh")
    num = np.trim_zeros(np.ravel(num), "f")
    return num / den[0], den / den[0]

In [ ]:
T = 1.0
num, den = zoh_model(T)
a, b = num                                 # G(z) = (a z + b) / (z^2 - (1+a) z + a)
print("1. G(z) for T = 1 s")
print(f"   numerator   {a:.4f} z + {b:.4f}      (e^-1 = {np.exp(-1):.4f}, 1 - 2e^-1 = {1 - 2*np.exp(-1):.4f})")
print(f"   denominator z^2 {den[1]:+.4f} z {den[2]:+.4f}  = (z - 1)(z - {np.exp(-1):.4f})")

In [ ]:
def G(z):
    return np.polyval(num, z) / np.polyval(den, z)

In [ ]:
def closed_loop_poles(K, num=num, den=den):
    """Roots of the characteristic polynomial den(z) + K num(z)."""
    return np.roots(np.polyadd(den, K * num))

In [ ]:
# -----------------------------------------------------------------------------------------
# 2. Root locus: zero, circle, breakaway, critical gain
# -----------------------------------------------------------------------------------------
z0 = -b / a                                # the zero of G(z), equal to 2 - e

In [ ]:
R = np.sqrt((1 - z0) * (a - z0))           # radius of the circle centered at the zero

In [ ]:
# breakaway/break-in: A'B - AB' = 0 with A = den, B = num
cond = np.polysub(np.polymul(np.polyder(den), num), np.polymul(den, np.polyder(num)))

In [ ]:
zb, zi = sorted(np.roots(cond).real, reverse=True)
K_of = lambda s: -np.polyval(den, s) / np.polyval(num, s)
Kcr = brentq(lambda K: max(abs(closed_loop_poles(K))) - 1, 1.0, 5.0)
theta = abs(np.angle(closed_loop_poles(Kcr)[0]))
print("\n2. Root locus")
print(f"   zero z0 = {z0:.4f} (2 - e = {2 - np.e:.4f}); circle radius R = {R:.4f}")
print(f"   breakaway {zb:.4f} at K = {K_of(zb):.3f};  break-in {zi:.4f} at K = {K_of(zi):.2f}")
print(f"   critical gain {Kcr:.3f}  ((e-1)/(e-2) = {(np.e - 1)/(np.e - 2):.3f}), "
      f"crossing angle {theta:.3f} rad, {2*np.pi/theta:.2f} samples per cycle")

In [ ]:
def zeta_wn(z, T=1.0):
    """Equivalent damping ratio and natural frequency of a z-plane pole (Chapter 10)."""
    r, th = abs(z), abs(np.angle(z))
    L = np.hypot(np.log(r), th)
    return -np.log(r) / L, L / T

In [ ]:
# -----------------------------------------------------------------------------------------
# 3. The gain for zeta = 0.5
# -----------------------------------------------------------------------------------------
K05 = brentq(lambda K: zeta_wn(closed_loop_poles(K)[0])[0] - 0.5, 0.2, 1.0)

In [ ]:
p05 = closed_loop_poles(K05)[0]
print("\n3. Target damping")
print(f"   zeta = 0.5 at K = {K05:.3f}: poles {p05.real:.3f} +- j{abs(p05.imag):.3f}, "
      f"wn = {zeta_wn(p05)[1]:.3f} rad/s, ramp error 1/K = {1/K05:.2f}")

In [ ]:
# Figure: the root locus with the unit circle and the zeta = 0.5 spiral
fig, ax = plt.subplots(figsize=(7, 5.5))

In [ ]:
Ks = np.concatenate([np.linspace(0, 0.2, 400), np.linspace(0.2, 40, 4000)])
locus = np.array([np.sort_complex(closed_loop_poles(K)) for K in Ks])
ax.plot(locus.real, locus.imag, ".", ms=1.5, color="crimson")
t = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.cos(t), np.sin(t), color="gray", lw=0.8)
ph = np.linspace(0, np.pi, 300)
c = 0.5 / np.sqrt(1 - 0.25)

In [ ]:
for s in (1, -1):
    ax.plot(np.exp(-c * ph) * np.cos(ph), s * np.exp(-c * ph) * np.sin(ph), "--", color="tab:blue", lw=0.8)

In [ ]:
ax.plot([1, a], [0, 0], "kx", ms=8, mew=2)
ax.plot([z0], [0], "ko", mfc="white", ms=7)
pc = closed_loop_poles(Kcr)
ax.plot(pc.real, pc.imag, "o", color="crimson")
ax.plot([p05.real] * 2, [p05.imag, -p05.imag], "o", color="green")
ax.set_xlim(-2.6, 1.4)
ax.set_ylim(-1.6, 1.6)
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_title("Root locus, ZOH + 1/(s(s+1)), T = 1 s")
ax.set_xlabel("Re z")
ax.set_ylabel("Im z")
fig.savefig(os.path.join(OUT, "ch12_root_locus.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
# -----------------------------------------------------------------------------------------
# 4. Effect of the sampling period on the critical gain
# -----------------------------------------------------------------------------------------
print("\n4. Critical gain against T")

In [ ]:
print("   T      zero     Kcr     2/T+1/3   theta    theta/T")

In [ ]:
for Ti in (0.1, 0.5, 1.0):
    n_i, d_i = zoh_model(Ti)
    Kc = brentq(lambda K: max(abs(closed_loop_poles(K, n_i, d_i))) - 1, 0.5, 100.0)
    th_i = abs(np.angle(closed_loop_poles(Kc, n_i, d_i)[0]))
    print(f"   {Ti:<5}  {-n_i[1]/n_i[0]:.3f}  {Kc:7.2f}  {2/Ti + 1/3:7.2f}   {th_i:.3f}    {th_i/Ti:.2f}")

In [ ]:
# -----------------------------------------------------------------------------------------
# 5. The w-plane: Gw(w) = G(z) with z = (1 + wT/2)/(1 - wT/2); Bode plot and margins
# -----------------------------------------------------------------------------------------
def Gw(w, T=1.0):
    return G((1 + w * T / 2) / (1 - w * T / 2))

In [ ]:
ww = np.logspace(-1.3, 2, 600)
resp = Gw(1j * ww)
mag_db = 20 * np.log10(abs(resp))
phase = np.degrees(np.unwrap(np.angle(resp)))
wwc = brentq(lambda x: abs(Gw(1j * x)) - 1, 0.1, 1.5)          # gain crossover
wwp = brentq(lambda x: Gw(1j * x).imag, 1.0, 3.0)              # phase crossover
PM = 180 + np.degrees(np.angle(Gw(1j * wwc)))
GM = 1 / abs(Gw(1j * wwp))
to_omega = lambda x: (2 / T) * np.arctan(x * T / 2)            # undo the warping
print("\n5. Margins at K = 1 (w-plane)")
print(f"   pole at w = -{2*(1-a)/(1+a):.4f} (2 tanh(1/2) = {2*np.tanh(0.5):.4f}), "
      f"zeros at w = {2/T:.0f} and w = -{2*(a+b)/(a-b):.2f}")
print(f"   PM = {PM:.1f} deg at ww = {wwc:.3f} (omega = {to_omega(wwc):.3f} rad/s)")
print(f"   GM = {GM:.3f} = {20*np.log10(GM):.2f} dB at ww = {wwp:.3f} (omega = {to_omega(wwp):.3f} rad/s)")
print(f"   GM equals the critical gain of the root locus: {np.isclose(GM, Kcr)}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax1.semilogx(ww, mag_db, color="crimson", label="sampled, Gw(j ww)")
ax1.semilogx(ww, -20 * np.log10(ww * np.sqrt(1 + ww**2)), "--", color="gray", label="analog Gp(j w)")
ax1.axhline(0, color="k", lw=0.6)
ax1.axvline(wwc, color="k", ls=":", lw=0.8)
ax1.axvline(wwp, color="k", ls=":", lw=0.8)
ax1.set_ylabel("magnitude (dB)")
ax1.set_ylim(-45, 30)
ax1.legend()
ax2.semilogx(ww, phase, color="crimson")
ax2.semilogx(ww, -90 - np.degrees(np.arctan(ww)), "--", color="gray")
ax2.axhline(-180, color="k", lw=0.6)
ax2.axvline(wwc, color="k", ls=":", lw=0.8)
ax2.axvline(wwp, color="k", ls=":", lw=0.8)
ax2.set_ylabel("phase (deg)")
ax2.set_xlabel("ww (rad/s)")
ax1.set_title(f"w-plane Bode plot, K = 1: PM = {PM:.1f} deg, GM = {20*np.log10(GM):.2f} dB")
fig.savefig(os.path.join(OUT, "ch12_bode_w.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
# -----------------------------------------------------------------------------------------
# 6. Nyquist plot and Z = N + P
# -----------------------------------------------------------------------------------------
def contour(eps=1e-3, m=200000):
    """Unit circle counterclockwise, with a small outward half-circle around z = 1."""
    t0 = 2 * np.arcsin(eps / 2)
    th = np.linspace(t0, 2 * np.pi - t0, m)
    phi = np.linspace(np.angle(np.exp(-1j * t0) - 1), np.angle(np.exp(1j * t0) - 1), 20000)
    return np.concatenate([np.exp(1j * th), 1 + eps * np.exp(1j * phi)])

In [ ]:
def clockwise_encirclements(curve, q):
    d = np.diff(np.angle(np.append(curve, curve[0]) - q))
    d = (d + np.pi) % (2 * np.pi) - np.pi
    return -int(round(d.sum() / (2 * np.pi)))

In [ ]:
Lgam = G(contour())
print("\n6. Nyquist: P = 0 (the pole at z = 1 is indented and counts as inside)")
print(f"   crossings of the negative real axis: {-1/Kcr:.3f} and G(-1) = {G(-1.0):.4f}")

In [ ]:
for K in (1.0, 10.0, 30.0):
    N = clockwise_encirclements(Lgam, -1 / K)
    Z = int(sum(abs(closed_loop_poles(K)) > 1))
    print(f"   K = {K:5.1f}: N = {N}, Z = N + P = {N}, poles outside the circle: {Z}")

In [ ]:
th = np.linspace(0.3, np.pi, 800)
Lh = G(np.exp(1j * th))
fig, ax = plt.subplots(figsize=(5.5, 6.5))
ax.plot(Lh.real, Lh.imag, color="crimson", label="0 < w < ws/2")
ax.plot(Lh.real, -Lh.imag, ":", color="crimson", label="mirror image")
ax.plot(np.cos(t), np.sin(t), color="gray", lw=0.8)
ax.plot([-1], [0], "k+", ms=10, mew=2)
ax.axvline(-1.5, color="gray", ls="--", lw=0.8)
ax.set_xlim(-2.0, 1.1)
ax.set_ylim(-2.2, 2.2)
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.legend(loc="upper right")
ax.set_title("Nyquist plot of G(e^{jwT}), T = 1 s")
fig.savefig(os.path.join(OUT, "ch12_nyquist.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
# -----------------------------------------------------------------------------------------
# 7. One sample of computation delay: D(z) = K/z
# -----------------------------------------------------------------------------------------
den_d = np.polymul(den, [1.0, 0.0])        # z (z - 1)(z - a)

In [ ]:
Kd = brentq(lambda K: max(abs(closed_loop_poles(K, num, den_d))) - 1, 0.3, 2.0)
pd = closed_loop_poles(Kd, num, den_d)
print("\n7. With one sample of delay")
print(f"   asymptote center {(1 + a - z0)/2:.3f}; critical gain {Kd:.3f}; "
      f"crossing angle {max(np.angle(pd[abs(pd.imag) > 1e-9])):.3f} rad; third pole {min(pd.real):.3f}")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
Ks = np.linspace(0, 8, 6000)
loc = np.array([np.sort_complex(closed_loop_poles(K, num, den_d)) for K in Ks])
ax.plot(loc.real, loc.imag, ".", ms=1.5, color="crimson")
ax.plot(np.cos(t), np.sin(t), color="gray", lw=0.8)
ax.plot([1, a, 0], [0, 0, 0], "kx", ms=8, mew=2)
ax.plot([z0], [0], "ko", mfc="white", ms=7)
ax.axvline((1 + a - z0) / 2, color="gray", ls="--", lw=0.8)
ax.set_xlim(-1.2, 1.5)
ax.set_ylim(-1.4, 1.4)
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_title("Root locus with D(z) = K/z, T = 1 s")
fig.savefig(os.path.join(OUT, "ch12_root_locus_delay.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"\nFigures written to {OUT}")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**Ride the root locus.** The running loop: a zero-order hold, $G_p(s) = 1/(s(s+1))$, gain $K$
and unity feedback. The gray curve is the root locus of $1 + K\,G(z) = 0$; the orange crosses are
the closed-loop poles at your $K$. At $T = 1$ s the poles break away from the real axis at
$K = 0.196$, cross the dashed $\zeta = 0.5$ curve at $K = 0.516$, and leave the unit circle at
$K = 2.39$. Shrink $T$ and the critical gain grows (about $2/T + 1/3$). Switch to $D(z) = K/z$ to
see what one sample of computation delay does: the limit drops to 0.805.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
NO_DELAY, DELAY = "D(z) = K", "D(z) = K/z"


def _loop(T, controller):
    """Numerator and denominator of the open loop D(z)G(z)/K, descending powers of z."""
    p = np.exp(-T)
    num = np.array([T - 1 + p, 1 - p - T * p])
    den = np.array([1.0, -(1 + p), p])
    if controller == DELAY:
        den = np.polymul(den, [1.0, 0.0])
    return num, den


def _poles(K, num, den):
    return np.roots(np.polyadd(den, K * num))


def explore(K=1.0, T=1.0, controller=NO_DELAY):
    num, den = _loop(T, controller)
    poles = _poles(K, num, den)
    rmax = np.max(np.abs(poles))

    lo, hi = 1e-6, 10.0 / T + 5.0                      # critical gain by bisection on max |z|
    for _ in range(50):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if np.max(np.abs(_poles(mid, num, den))) < 1 else (lo, mid)
    k_cr = lo

    Ks = np.geomspace(1e-3, 60.0 / T, 900)
    locus = np.concatenate([_poles(k, num, den) for k in Ks])

    fig, ax = plt.subplots(figsize=(7.2, 5.6))
    th = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(th), np.sin(th), color="k", lw=0.9)
    ph = np.linspace(0, np.pi, 300)                    # zeta = 0.5: z = exp(-zeta/sqrt(1-zeta^2) theta + j theta)
    rr = np.exp(-ph / np.sqrt(3))
    for sgn in (1, -1):
        ax.plot(rr * np.cos(ph), sgn * rr * np.sin(ph), "--", color=TEAL, lw=1.2,
                label="$\\zeta = 0.5$" if sgn == 1 else None)
    ax.plot(locus.real, locus.imag, ".", color=GRAY, ms=2.2)
    op = np.roots(den)
    ax.plot(op.real, op.imag, "x", color=BLUE, ms=10, mew=2, label="open-loop poles")
    z0 = -num[1] / num[0]
    ax.plot([z0], [0], "o", mfc="white", mec=BLUE, mew=2, ms=9, label="zero")
    ax.plot(poles.real, poles.imag, "X", color=ORANGE, ms=13, mec="k", mew=0.8,
            label=f"closed-loop poles, K = {K:g}")
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    ax.set_xlim(-2.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.set_aspect("equal")
    ax.set_xlabel("Re z")
    ax.set_ylabel("Im z")
    ax.set_title(f"{controller}, T = {T:g} s: critical gain {k_cr:.3f}", fontsize=11)
    ax.legend(fontsize=10, loc="lower left")
    fig.tight_layout()

    print("closed-loop poles:", ", ".join(f"{z.real:.3f}{z.imag:+.3f}j" for z in poles),
          f"   max |z| = {rmax:.3f}  ->  " + ("stable" if rmax < 1 else "NOT stable"))
    zp = poles[np.argmax(poles.imag)]
    if abs(zp.imag) > 1e-9 and abs(zp) < 1:
        s = np.log(zp) / T
        print(f"complex pair: zeta = {-s.real / abs(s):.3f}, wn = {abs(s):.3f} rad/s")
    print(f"critical gain at T = {T:g} s: K = {k_cr:.3f};  zero of G(z) at z = {z0:.3f}")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(explore,
             K=FloatSlider(value=1.0, min=0.02, max=10.0, step=0.02, description="K",
                           continuous_update=False),
             T=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.05, description="T (s)",
                           continuous_update=False),
             controller=Dropdown(options=[NO_DELAY, DELAY], value=NO_DELAY, description="D(z)"))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 12.** These cells go with the Hands-on exercises at the end of
Chapter 12. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def locus(A, B, Ks):
    """Roots of A(z) + K B(z) for each K (A, B as coefficient lists, highest power first)."""
    n = len(A) - 1
    Bp = np.r_[np.zeros(n + 1 - len(B)), B]
    return np.array([np.sort_complex(np.roots(np.asarray(A, float) + K * Bp)) for K in Ks])

In [ ]:
def unit_circle(ax, lim=1.4):
    c = np.linspace(0, 2 * np.pi, 300)
    ax.plot(np.cos(c), np.sin(c), color=GRAY, lw=1)
    ax.axhline(0, color=GRAY, lw=0.5)
    ax.axvline(0, color=GRAY, lw=0.5)
    ax.set_aspect("equal")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

In [ ]:
# ---------------------------------------------------------------- H1
def h1_heater_locus(K=1.0):
    """H1: D(z)G(z) = 0.2K/((z-1)(z-0.6)), T = 0.5 s; locus, current poles, step response."""
    T = 0.5
    A, B = [1, -1.6, 0.6], [0.2]
    R = locus(A, B, np.linspace(0, 2.5, 600))
    p = np.roots(np.r_[1, -1.6, 0.6 + 0.2 * K])
    r = max(abs(p))
    tau = -T / np.log(r) if r < 1 else np.inf
    n = 40
    y = np.zeros(n + 1)
    for k in range(2, n + 1):           # closed loop 0.2K / (z^2 - 1.6 z + 0.6 + 0.2K), unit step
        y[k] = 1.6 * y[k - 1] - (0.6 + 0.2 * K) * y[k - 2] + 0.2 * K
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios": [1, 1.6]})
    unit_circle(ax1)
    ax1.plot(R.real, R.imag, ".", color=BLUE, ms=1.5)
    ax1.plot(p.real, p.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax1.set_title(f"K = {K:.2f}: |z| = {r:.3f}", fontsize=11)
    ax2.plot(T * np.arange(n + 1), y, "o-", color=ORANGE, ms=3, lw=1)
    ax2.axhline(1, color=GRAY, lw=0.6)
    ax2.set_ylim(-0.2, 2.2)
    ax2.set_xlabel("t (s)")
    ax2.set_title(f"step response, envelope tau = {tau:.2f} s", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_gimbal_bode(K=4.0):
    """H2: Gw(w) = K (1 - w/20)(1 + w/241) / (w (1 + w/4.90)), T = 0.1 s; correct vs sign-flipped phase."""
    ww = np.logspace(-1, 3, 3000)
    s = 1j * ww
    Gw = K * (1 - s / 20) * (1 + s / 241) / (s * (1 + s / 4.90))
    ph = np.degrees(np.unwrap(np.angle(Gw)))
    ph_wrong = -90 - np.degrees(np.arctan(ww / 4.90)) + np.degrees(np.arctan(ww / 20)) \
        + np.degrees(np.arctan(ww / 241))
    ic = np.argmin(abs(abs(Gw) - 1))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True)
    ax1.semilogx(ww, 20 * np.log10(abs(Gw)), color=BLUE, lw=1.4)
    ax1.axhline(0, color=GRAY, lw=0.6)
    ax1.set_ylabel("|Gw| (dB)")
    ax1.set_title(f"K = {K:g}: crossover w = {ww[ic]:.2f} rad/s, "
                  f"PM {180 + ph[ic]:.1f} deg (sign-flipped reading {180 + ph_wrong[ic]:.1f} deg)",
                  fontsize=11)
    ax2.semilogx(ww, ph, color=BLUE, lw=1.4, label="correct phase")
    ax2.semilogx(ww, ph_wrong, "--", color=ORANGE, lw=1.4, label="zero 1 - w/20 read as a lead")
    ax2.axhline(-180, color=GRAY, lw=0.6)
    ax2.set_ylabel("phase (deg)")
    ax2.set_xlabel("w-plane frequency (rad/s)")
    ax2.legend(loc="lower left", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_reactor(K=1.7, T=1.0):
    """H3: Gp = a/(s - a), a = ln 2, ZOH, gain K: G(z) = (e^{aT} - 1)/(z - e^{aT})."""
    a = np.log(2)
    E = np.exp(a * T)
    pole = E - K * (E - 1)
    w = np.linspace(0, 2 * np.pi, 800)
    L = K * (E - 1) / (np.exp(1j * w) - E)
    ang = np.unwrap(np.angle(L + 1))
    ccw = int(np.round((ang[-1] - ang[0]) / (2 * np.pi)))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
    unit_circle(ax1, lim=2.6)
    Ks = np.linspace(0, 4, 200)
    ax1.plot(E - Ks * (E - 1), 0 * Ks, color=BLUE, lw=2)
    ax1.plot([E], [0], "x", color=GRAY, ms=10, mew=2)
    ax1.plot([pole], [0], "x", color=ORANGE, ms=12, mew=2.5)
    ax1.set_title(f"T = {T:.2f} s, K = {K:.2f}: pole {pole:.2f} "
                  f"({'stable' if abs(pole) < 1 else 'unstable'})", fontsize=11)
    ax2.plot(L.real, L.imag, color=BLUE, lw=1.4)
    ax2.plot([-1], [0], "+", color=ORANGE, ms=14, mew=2.5)
    ax2.set_aspect("equal")
    ax2.axhline(0, color=GRAY, lw=0.5)
    ax2.axvline(0, color=GRAY, lw=0.5)
    ax2.set_title(f"Nyquist of K G: encircles -1 {ccw:+d} times (counterclockwise +)", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_resonance(alpha=0.7):
    """H4: G = 0.1/((z-p)(z-conj p)), p = 0.8 + j0.5, D = K (z - alpha)/z; the locus near p."""
    p = 0.8 + 0.5j
    A = np.polymul([1, 0], [1, -1.6, 0.89])           # z (z^2 - 1.6 z + 0.89)
    B = [0.1, -0.1 * alpha]
    R = locus(A, B, np.linspace(0, 12, 1500))
    th_d = 180 + np.degrees(np.angle(p - alpha)) - np.degrees(np.angle(p)) - 90
    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    unit_circle(ax, lim=1.3)
    c = np.linspace(0, 2 * np.pi, 300)
    ax.plot(abs(p) * np.cos(c), abs(p) * np.sin(c), ":", color=TEAL, lw=1)
    ax.plot(R.real, R.imag, ".", color=BLUE, ms=1.5)
    ax.plot([p.real, p.real, 0], [p.imag, -p.imag, 0], "x", color=GRAY, ms=10, mew=2)
    ax.plot([alpha], [0], "o", mfc="white", color=GRAY, ms=8)
    rad = np.radians(np.degrees(np.angle(p)))
    ax.annotate("", xy=(p.real + 0.25 * np.cos(rad), p.imag + 0.25 * np.sin(rad)),
                xytext=(p.real, p.imag), arrowprops=dict(arrowstyle="->", color=GRAY))
    ax.annotate("", xy=(p.real + 0.25 * np.cos(np.radians(th_d)), p.imag + 0.25 * np.sin(np.radians(th_d))),
                xytext=(p.real, p.imag), arrowprops=dict(arrowstyle="->", color=ORANGE, lw=2))
    diff = (th_d - np.degrees(np.angle(p))) % 360
    ax.set_title(f"alpha = {alpha:.2f}: departure {th_d:.1f} deg, {diff:.1f} deg from radial "
                 f"({'inward' if 90 < diff < 270 else 'outward'})", fontsize=10)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    interact(h1_heater_locus, K=FloatSlider(value=1.0, min=0.02, max=2.4, step=0.02,
                                            description="K", continuous_update=False))
    interact(h2_gimbal_bode, K=FloatSlider(value=4.0, min=1.0, max=20.0, step=0.5,
                                           description="K", continuous_update=False))
    interact(h3_reactor, K=FloatSlider(value=1.7, min=0.2, max=4.0, step=0.05, description="K",
                                       continuous_update=False),
             T=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.05, description="T (s)",
                           continuous_update=False))
    interact(h4_resonance, alpha=FloatSlider(value=0.7, min=-0.5, max=0.95, step=0.01,
                                             description="alpha", continuous_update=False))
except ImportError:
    h1_heater_locus()
    h2_gimbal_bode()
    h3_reactor()
    h4_resonance()